# Embedding Types

**Module:** 01 — Embeddings

Not all embeddings are interchangeable. This lesson maps query, document, chunk, sentence, and multimodal embedding types to the jobs they do.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Differentiate query vs document vs chunk embeddings
- Explain sentence vs document tradeoffs
- Describe image/audio/video and multimodal embedding use cases
- Pick an embedding type for a concrete product scenario


## Query Embeddings

**Definition.** Vectors produced from user questions or search strings at request time.

**Why it matters.** Query encoders may be tuned for short, interrogative text and asymmetric retrieval.

**How it works.** Embed the query with the matching query tower / instruction prefix (e.g., E5 'query:').

**Intuition.** Queries are flashlights; documents are objects in a dark room.

**Common pitfalls.**
- Using the document prefix on queries for asymmetric models
- Caching query vectors that contain personal data without TTL/policy

**When to use.** Online path of every semantic search / RAG request.


In [ ]:
def format_e5(text, is_query=True):
    return ("query: " if is_query else "passage: ") + text

print(format_e5("What is the refund window?", True))
print(format_e5("Refunds are available for 14 days.", False))


### Try it yourself — Query Embeddings

1. Write a one-sentence product use case that needs query embeddings.
2. Name one metric you would track in production for this embedding type.


## Document Embeddings

**Definition.** Vectors representing a full document unit (article, ticket, page).

**Why it matters.** Useful for doc-level navigation, but coarse for grounded QA.

**How it works.** Encode whole docs or aggregate chunk vectors (mean/max/learned pooling).

**Intuition.** A book blurb vs every paragraph — blurbs help browsing, paragraphs help answers.

**Common pitfalls.**
- One vector per huge PDF for precise QA

**When to use.** Doc routing, clustering, site-level search.


In [ ]:
import numpy as np
chunk_vecs = np.random.default_rng(0).normal(size=(4, 8))
doc_vec = chunk_vecs.mean(axis=0)
print(doc_vec.shape, np.round(doc_vec, 3))


### Try it yourself — Document Embeddings

1. Write a one-sentence product use case that needs document embeddings.
2. Name one metric you would track in production for this embedding type.


## Chunk Embeddings

**Definition.** Vectors for retrieval units cut from documents (paragraphs, 200–800 tokens, etc.).

**Why it matters.** Core of modern RAG: enough context to answer, small enough to stay specific.

**How it works.** Chunk offline → embed → index with parent doc metadata.

**Intuition.** Index recipe cards, not entire cookbooks.

**Common pitfalls.**
- Chunks that split mid-table or mid-code block
- No overlap when references cross boundaries

**When to use.** Default choice for RAG knowledge bases.


In [ ]:
def chunk_words(text, size=6, overlap=2):
    toks = text.split()
    out, i = [], 0
    while i < len(toks):
        out.append(" ".join(toks[i:i+size]))
        i += max(1, size - overlap)
    return out

print(chunk_words("Acme refunds are available within fourteen days of purchase for unused items"))


### Try it yourself — Chunk Embeddings

1. Write a one-sentence product use case that needs chunk embeddings.
2. Name one metric you would track in production for this embedding type.


## Sentence Embeddings

**Definition.** Vectors for single sentences or short passages.

**Why it matters.** Ideal for FAQ matching, bitext mining, and clustering short texts.

**How it works.** Use sentence-transformer style encoders with mean/CLS pooling + normalize.

**Intuition.** Business cards for sentences—compact identity.

**Common pitfalls.**
- Sentence vectors for long multi-topic answers

**When to use.** FAQ bots, semantic dedup of short messages, evaluation datasets.


In [ ]:
sentences = ["Reset your password", "Change account credentials", "Track my shipment"]
# Toy: character-ngram hashing embedder
import numpy as np

def sent_embed(s, dim=32):
    v = np.zeros(dim)
    s = s.lower()
    for i in range(len(s)-2):
        v[hash(s[i:i+3]) % dim] += 1
    return v / (np.linalg.norm(v)+1e-9)

q = sent_embed(sentences[0])
for s in sentences:
    print(s, round(float(np.dot(q, sent_embed(s))), 3))


### Try it yourself — Sentence Embeddings

1. Write a one-sentence product use case that needs sentence embeddings.
2. Name one metric you would track in production for this embedding type.


## Image Embeddings

**Definition.** Vectors from vision encoders (CLIP-style) representing image content.

**Why it matters.** Enable reverse image search, visual dedup, and text↔image retrieval.

**How it works.** Encode images with a vision tower; optionally share space with text.

**Intuition.** A photo of a red sneaker should land near the text 'red running shoe'.

**Common pitfalls.**
- Domain gap: product shots vs user photos
- Ignoring OCR text present in images

**When to use.** Catalog search, moderation triage, multimodal RAG.


In [ ]:
# Pseudo CLIP scores (text-image) with placeholder vectors
import numpy as np
text = np.array([0.1, 0.8, 0.2]); text /= np.linalg.norm(text)
img_shoe = np.array([0.12, 0.75, 0.18]); img_shoe /= np.linalg.norm(img_shoe)
img_car = np.array([0.9, 0.1, 0.0]); img_car /= np.linalg.norm(img_car)
print("text↔shoe", round(float(np.dot(text, img_shoe)), 3))
print("text↔car ", round(float(np.dot(text, img_car)), 3))


### Try it yourself — Image Embeddings

1. Write a one-sentence product use case that needs image embeddings.
2. Name one metric you would track in production for this embedding type.


## Audio Embeddings

**Definition.** Vectors summarizing audio segments (speech, music, events).

**Why it matters.** Power voice search, speaker clustering, and audio event retrieval.

**How it works.** Segment audio → encode with audio model → index; text queries may use aligned text space.

**Intuition.** A fingerprint for how a clip sounds/means, not the raw waveform.

**Common pitfalls.**
- Segments too long mixing multiple speakers/topics

**When to use.** Call-center analytics, podcast search, multimodal agents.


In [ ]:
# Frame aggregation sketch
import numpy as np
frames = np.random.default_rng(1).normal(size=(50, 16))  # fake frame features
clip_emb = frames.mean(axis=0)
clip_emb = clip_emb / (np.linalg.norm(clip_emb)+1e-9)
print(clip_emb.shape)


### Try it yourself — Audio Embeddings

1. Write a one-sentence product use case that needs audio embeddings.
2. Name one metric you would track in production for this embedding type.


## Video Embeddings

**Definition.** Vectors for clips or keyframes, sometimes fused with audio/ASR text.

**Why it matters.** Needed for moment retrieval and video corpus search.

**How it works.** Sample frames / shots → encode → pool temporally; store timestamps as metadata.

**Intuition.** Search for 'moments', not entire movies, unless browsing.

**Common pitfalls.**
- One embedding per full movie for fine-grained queries

**When to use.** Media archives, course libraries, safety review.


In [ ]:
# Keyframe pooling
import numpy as np
keyframes = np.random.default_rng(2).normal(size=(8, 16))
video_emb = keyframes.mean(axis=0)
print(video_emb.shape)


### Try it yourself — Video Embeddings

1. Write a one-sentence product use case that needs video embeddings.
2. Name one metric you would track in production for this embedding type.


## Multimodal Embeddings

**Definition.** Shared or aligned spaces where text, image, audio can be compared directly.

**Why it matters.** One query modality can retrieve another (text→image, image→text).

**How it works.** Contrastive training aligns towers; at query time embed each modality with its tower.

**Intuition.** Universal translators into a shared meaning space.

**Common pitfalls.**
- Assuming perfect cross-modal calibration out of the box
- Evaluating only text→text

**When to use.** Catalogs, multimodal RAG, creative search tools.


In [ ]:
modalities = {"text": [0.1, 0.8], "image": [0.12, 0.76], "audio": [0.7, 0.2]}
import numpy as np
for k,v in modalities.items():
    x = np.array(v, float); x /= np.linalg.norm(x); modalities[k]=x
q = modalities["text"]
for k,v in modalities.items():
    print(k, round(float(np.dot(q,v)), 3))


### Try it yourself — Multimodal Embeddings

1. Write a one-sentence product use case that needs multimodal embeddings.
2. Name one metric you would track in production for this embedding type.


## Comparison table

| Type | Granularity | Typical app | Risk if misused |
|---|---|---|---|
| Query | Request-time text | Search / RAG | Wrong instruction prefix |
| Document | Whole doc | Browse / route | Weak grounded answers |
| Chunk | Passage | RAG default | Boundary / context loss |
| Sentence | One sentence | FAQ | Oversimplifies long answers |
| Image/Audio/Video | Media segment | Multimodal search | Domain gap |
| Multimodal | Shared space | Cross-modal retrieval | Misaligned towers |


<!-- enriched:v1 -->

### Choosing Granularity

| Unit | Best for | Risk |
|------|----------|------|
| Query | Asymmetric retrieval | Must match training prefixes |
| Sentence | FAQ, short QA | Misses multi-sentence constraints |
| Chunk | RAG | Boundary artifacts |
| Document | Clustering, routing | Diluted semantics |

In [ ]:
# Simulate query vs doc asymmetry with different 'encoders'
import numpy as np

def enc(text, salt=0):
    rng = np.random.RandomState((abs(hash(text)) + salt) % (2**32))
    v = rng.randn(16); return v / np.linalg.norm(v)

q = "password reset"
# Symmetric encoder (same salt)
docs = ["reset your password via email", "shipping to Canada"]
print("Symmetric scores:", [round(float(enc(q) @ enc(d)), 3) for d in docs])
# Asymmetric: query and passage use different projection salts (toy)
print("Asymmetric scores:", [round(float(enc(q,1) @ enc(d,2)), 3) for d in docs])

### Image / Audio / Video Pipelines

1. **Image:** preprocess → vision encoder → optional text alignment (CLIP)
2. **Audio:** waveform/mel → audio encoder → embedding
3. **Video:** sample frames (+ audio) → temporal pool → clip embedding

Always store **modality + model_version** in metadata.

In [ ]:
# Frame sampling sketch for video embeddings
fps_sample = 1  # 1 frame per second
duration_s = 12
frame_indices = list(range(0, duration_s * 30, 30 // fps_sample))  # assume 30fps source
print("Sampled frame indices:", frame_indices[:10], "...")
print("Aggregate with mean/NetVLAD/attention pooling over frame vectors.")

### Multimodal Retrieval Pattern

Text query → embed in shared space → search image index (or vice versa). For RAG over PDFs, embed text chunks **and** page images; fuse ranks.

## Summary & Key Takeaways

- Match embedding type to retrieval granularity.
- Chunk embeddings are the workhorse of RAG.
- Asymmetric models need correct query/document formatting.
- Multimodal spaces unlock cross-modal product experiences.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
